# Text Classification: Fine-tuning BERT

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Install dependencies

Run once if these aren't already installed.

In [ ]:
# !pip install torch transformers datasets scikit-learn pandas

## 1. Imports

In [2]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## 2. Download a public dataset and 3. save it to CSV

We use **AG News**, a public dataset with 4 topic classes: `World (0)`, `Sports (1)`, `Business (2)`, `Sci/Tech (3)`.

To keep training fast, we take a small subset. Increase the sizes for better accuracy.

## 4. Read the data back from the CSV files

In [4]:
train_df = pd.read_csv("/content/drive/MyDrive/MLDatasets/text_classification/train_10k.csv").sample(2000)
test_df = pd.read_csv("/content/drive/MyDrive/MLDatasets/text_classification/test_2k.csv").sample(500)

label_names = ["World", "Sports", "Business", "Sci/Tech"]
num_labels = len(label_names)

print(train_df.shape, test_df.shape)
train_df.head()

(2000, 3) (500, 3)


,label,heading,text
2522,2,Update 14: Google Lowers Its IPO Price Range,In a sign that Google Inc. #39;s initial publi...
30808,2,Daily Briefing,EUGENE -- Recreational vehicle maker Country C...
37039,0,Iraq Leader Tells Congress That Sacrifice Won'...,"Offering a simple, ""Thank you America,"" Iraqi ..."
13242,2,US Economy Grew at Slower Pace Than Expected,he United States economy grew more slowly in t...
6278,2,Lawsuits hold good news for investors,If you are looking for the most interesting fa...


## 5. Tokenize

We wrap the tokenized data in a tiny `torch.utils.data.Dataset`.

In [5]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        # encodings -> token_ids, token_type_ids, attention_mask
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len,
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_ds = TextDataset(train_df["text"], train_df["label"], tokenizer)
test_ds = TextDataset(test_df["text"], test_df["label"], tokenizer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
train_ds[12]

## input_ids -> token_ids
## token_id for padding token [PAD] is 0 in this tokenizer
## token_type_ids -> these are useful in seq-seq task
## attention_mask -> 0 for [PAD] as we don't want [PAD] to be part of attention computations as they are not part of the text

{'input_ids': tensor([  101,  3190,  1006, 26665,  1007,  1011,  2436,  8470,  4297,  1012,
          1004,  8318,  1025,  1037, 17850, 12879,  1027,  1000,  8299,  1024,
          1013,  1013,  7479,  1012, 14316,  1012, 26665,  1012,  4012,  1013,
          2440, 28940, 12184,  1012,  2004,  2361,  2595,  1029, 16356,  2121,
          1027,  1051, 18927,  1012,  1050,  4539,  1027,  1013, 15768,  1013,
          4248,  2378, 14876,  1013,  2440, 28940, 12184,  1000,  1004, 14181,
          1025,  1051, 18927,  1012,  1050,  1004,  8318,  1025,  1013,  1037,
          1004, 14181,  1025,  2006,  9857,  7420,  2008,  5618,  2005,  1996,
          2717,  1997,  1996,  2095,  2052,  2991,  2460,  1997, 19939,  2015,
          2138,  1997,  1996,  5164,  1997, 17035,  1999,  3516,  1998,  3532,
          4341,  6974,  1010,  6016,  2049,  4518,  2091,  1020,  1012,  1019,
          3867,  1012,   102,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [12]:
tokens = tokenizer.convert_ids_to_tokens(train_ds[8]['input_ids'])
print(tokens)

['[CLS]', 'the', 'last', 'time', 'apple', 'computer', 'inc', '.', 'introduced', 'an', 'all', '-', 'new', 'im', '##ac', 'computer', ',', 'back', 'in', 'january', '2002', ',', 'apple', 'chief', 'executive', 'steve', 'jobs', 'was', 'featured', 'smiling', 'with', 'the', 'desktop', 'on', 'the', 'cover', 'of', 'time', 'magazine', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[P

In [14]:
id2label={i: name for i, name in enumerate(label_names)}
label2id={name: i for i, name in enumerate(label_names)}

print(id2label)
print(label2id)

{0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
{'World': 0, 'Sports': 1, 'Business': 2, 'Sci/Tech': 3}


## 6. Fine-tune (training)

In [15]:
# AutoModelForSequenceClassification
# Downloads the given transformer model , adds the custom classification layer on top of it

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    # Bake the class names into the model config so they get saved and
    # every loader (including `pipeline`) reports real names, not LABEL_0/1/...
    id2label={i: name for i, name in enumerate(label_names)},
    label2id={name: i for i, name in enumerate(label_names)},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir="./bert-output",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.395167,0.390011,0.868000,0.867578
2,0.320600,0.347169,0.872000,0.871727


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=250, training_loss=0.5188763580322265, metrics={'train_runtime': 102.6611, 'train_samples_per_second': 38.963, 'train_steps_per_second': 2.435, 'total_flos': 263115780096000.0, 'train_loss': 0.5188763580322265, 'epoch': 2.0})

## 7. Evaluate

In [16]:
metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.320600,0.347169,2,0.872000,0.871727


{'eval_loss': 0.34716928005218506, 'eval_accuracy': 0.872, 'eval_f1': 0.8717274976297048}


## 8. Save the fine-tuned model

In [17]:
trainer.save_model("./bert-text-classifier")
tokenizer.save_pretrained("./bert-text-classifier")
print("Model saved to ./bert-text-classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bert-text-classifier


## 9. Inference on new text

In [18]:
def predict(texts):
    model.eval()
    enc = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        logits = model(**enc).logits
        # logits are last layer output from the classification layer
        # if num_classes = 3 , there will be 3 logits generated by classification layer
        # 1 logit per class is generated
        # logits => activations from last layer
        # classification layer activations output
    preds = logits.argmax(dim=-1).tolist()
    # select class with maximum logit value/probability
    return [label_names[p] for p in preds]


samples = [
    "The team clinched the championship in the final seconds of the game.",
    "The central bank raised interest rates to curb inflation.",
    "Researchers unveiled a new chip that doubles processing speed.",
    "Diplomats met to negotiate a ceasefire agreement.",
]

for text, pred in zip(samples, predict(samples)):
    print(f"[{pred}] {text}")

[Sports] The team clinched the championship in the final seconds of the game.
[Business] The central bank raised interest rates to curb inflation.
[Sci/Tech] Researchers unveiled a new chip that doubles processing speed.
[World] Diplomats met to negotiate a ceasefire agreement.


In [19]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="./bert-text-classifier",
    device=0 if device == "cuda" else -1,
)

for text in samples:
    print(clf(text), text)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'Sports', 'score': 0.9702802300453186}] The team clinched the championship in the final seconds of the game.
[{'label': 'Business', 'score': 0.9451993107795715}] The central bank raised interest rates to curb inflation.
[{'label': 'Sci/Tech', 'score': 0.9498481750488281}] Researchers unveiled a new chip that doubles processing speed.
[{'label': 'World', 'score': 0.9664759039878845}] Diplomats met to negotiate a ceasefire agreement.


In [20]:
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,